In [43]:
from pcamarillor.spark_utils import SparkUtils
from pyspark.sql import functions as F
import argparse

In [44]:
su = SparkUtils("Car Records", 
                "spark://spark-master:7077")
su.spark

In [18]:
schema = SparkUtils.generate_schema([
    ("id", "int"),
    ("license_plate", "string"),
    ("registration_date", "date"),
    ("year", "int"),
    ("model", "string"),
    ("brand", "string"),
    ("color", "string"),
    ("motor_no", "string"),
    ("type", "string"),
    ("firstname", "string"),
    ("lastname", "string"),
    ("address", "string"),
    ("municipality", "string"),
    ("state", "string")
])

car_records_path = "/opt/spark/work-dir/src/project/data"

car_records_df = su.spark.read \
                    .option("header", "true") \
                    .schema(schema) \
                    .csv(car_records_path)

car_records_df.printSchema()
car_records_df.show(5)

root
 |-- id: integer (nullable = true)
 |-- license_plate: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- model: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- color: string (nullable = true)
 |-- motor_no: string (nullable = true)
 |-- type: string (nullable = true)
 |-- firstname: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- address: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- state: string (nullable = true)



+---+-------------+-----------------+----+--------+----------+------+----------+------+---------+--------+--------------------+--------------------+---------------+
| id|license_plate|registration_date|year|   model|     brand| color|  motor_no|  type|firstname|lastname|             address|        municipality|          state|
+---+-------------+-----------------+----+--------+----------+------+----------+------+---------+--------+--------------------+--------------------+---------------+
|  0|     CTL-8626|       2016-11-25|2005|    Golf|Volkswagen|   Red|EA26GB38PI|Pickup|  Nicolás|Figueroa|Privada Norte Oca...|San Jacobo de la ...|         Puebla|
|  1|     NYD-5238|       2010-03-17|2026|  Accord|     Honda|  Gray|RY24QG75MJ|   SUV|     Abel|    Mora|Pasaje Sinaloa 52...|       Nueva Ucrania|        Tabasco|
|  2|     TMU-2210|       2006-11-19|2002|Frontier|    Nissan| Black|JB32CN64WX|   Van|      Jos| Collazo|Calzada Países Ba...|San Ernesto los b...|Baja California|
|  3|     

In [42]:
print(f"Total records before cleaning: {car_records_df.count()}")

# Drop duplicates based on the license_plate column
car_records_df.dropDuplicates(["license_plate"])

# Delete records where registration_date is before the year of the car
# clean_car_records_df = car_records_df.filter(F.year(F.col("registration_date")) >= F.col("year"))
filtered_car_records_df = car_records_df.filter(F.year(F.col("registration_date")) >= F.col("year")).filter(F.col("year") > 2010)

print(f"Total records after cleaning: {clean_car_records_df.count()}")

Total records before cleaning: 1000
Total records after cleaning: 473


In [40]:
records_after_2010 = clean_car_records_df.filter(F.col("year") > 2010)
print(f"Cars registered after 2010: {records_after_2010.count()}")
records_after_2010.show(5)

Cars registered after 2010: 144
+---+-------------+-----------------+----+------+----------+------+----------+------+---------+--------+--------------------+--------------------+----------------+
| id|license_plate|registration_date|year| model|     brand| color|  motor_no|  type|firstname|lastname|             address|        municipality|           state|
+---+-------------+-----------------+----+------+----------+------+----------+------+---------+--------+--------------------+--------------------+----------------+
|  5|     GDP-0032|       2022-01-21|2014| Yaris|    Toyota| Black|XJ49BS02OK|Pickup| Gabriela|  Segura|Circuito Valencia...|San María Cristin...| San Luis Potosí|
|  9|     OGN-2043|       2014-11-13|2011|Accord|     Honda| Black|IH73FS65PE|   Van|Ana Luisa|Ulibarri|Pasaje Guerrero 2...|San Leonel de la ...|Distrito Federal|
| 10|     WXM-7937|       2025-08-20|2018|Tiguan|Volkswagen|   Red|DE53OM44AN|Pickup|José Luis|   Baeza|Callejón Carrero ...|San Abelardo de l...|  

In [41]:
cars_by_state_df = records_after_2010.groupBy("state").count().orderBy(F.desc("count"))
cars_by_state_df.show(10)

+--------------------+-----+
|               state|count|
+--------------------+-----+
|             Jalisco|   10|
|            Tlaxcala|   10|
|    Distrito Federal|    8|
|          Guanajuato|    8|
|      Aguascalientes|    7|
|        Quintana Roo|    7|
|     Baja California|    7|
|Veracruz de Ignac...|    6|
|           Querétaro|    6|
|             Chiapas|    5|
+--------------------+-----+
only showing top 10 rows


Cars registered before 2000: 3
+---+-------------+-----------------+----+--------+---------+------+----------+---------+----------+--------+--------------------+--------------------+----------+
| id|license_plate|registration_date|year|   model|    brand| color|  motor_no|     type| firstname|lastname|             address|        municipality|     state|
+---+-------------+-----------------+----+--------+---------+------+----------+---------+----------+--------+--------------------+--------------------+----------+
| 60|     OTT-0658|       1998-05-13|1998|Frontier|   Nissan| Green|VL83NB62KB|Hatchback|     Laura| Vallejo|Retorno Armenia 8...|Nueva República D...|   Sinaloa|
| 29|     OLH-2898|       1998-09-27|1998|   Spark|Chevrolet|  Blue|AZ45HL45KZ|      Van|José María|   Parra|Eje vial Hungría ...|        Vieja Brasil|Tamaulipas|
| 54|     ZVW-5105|       1999-01-11|1999|     Fit|    Honda|Silver|MC07WA69KJ|      Van|   Silvano| Herrera|Callejón Guerrero...|San Armando de la...|   